# 2 · The provider publishes the factory

The provider describes what it shares in its catalog (DCAT), says on which terms (ODRL policies) and tells its agent how to reach the backend behind it (connectors). Here the backend is the demo factory: robots with yearly measures, and the events they emit in real time.

Every run of this notebook adds a new dataset to the provider's catalog; the next notebooks use the one it just created.

In [ ]:
%run ./01_auth.ipynb

In [ ]:
factory_api = env("EUNOMIA_FACTORY_DOCKER_URL")             # GET /robots/
factory_events = env("EUNOMIA_FACTORY_EVENTS_DOCKER_URL")   # /subscriptions
factory_token_url = env("EUNOMIA_FACTORY_TOKEN_DOCKER_URL") # Keycloak, OAuth2 password grant

## Catalog and data service

Every agent comes with a main catalog and a main data service: its DSP endpoint.

In [ ]:
response = requests.get(provider + "/api/v1/catalog-agent/catalogs/main", headers=json_header)
catalog_id = show(response, "The main catalog")["id"]

In [ ]:
response = requests.get(provider + "/api/v1/catalog-agent/data-services/main", headers=json_header)
data_service = show(response, "The main data service")
data_service_id = data_service["id"]

## Dataset and distributions

One dataset, two ways to get it: a pull distribution (a snapshot on demand) and a push one (a stream of events).

In [ ]:
payload = {
    "dctTitle": "Factory Robots (notebook)",
    "dctDescription": "The robots of the demo factory and their yearly measures, plus the real-time events they emit.",
    "catalogId": catalog_id,
}
response = requests.post(provider + "/api/v1/catalog-agent/datasets", headers=json_header, data=json.dumps(payload))
dataset_id = show(response, "New dataset")["id"]

In [ ]:
payload = {
    "dctTitle": "Factory Robots - HTTP Pull",
    "dctDescription": "On-demand snapshot via GET /robots/.",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "HTTP_PULL",
}
response = requests.post(provider + "/api/v1/catalog-agent/distributions", headers=json_header, data=json.dumps(payload))
distribution_pull_id = show(response, "Pull distribution")["id"]

In [ ]:
payload = {
    "dctTitle": "Factory Robots - HTTP Push",
    "dctDescription": "Real-time robot events, pushed to the consumer's endpoint.",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "HTTP_PUSH",
}
response = requests.post(provider + "/api/v1/catalog-agent/distributions", headers=json_header, data=json.dumps(payload))
distribution_push_id = show(response, "Push distribution")["id"]

## Policies

An ODRL policy is an offer: what the consumer may do with the data, must do, and must not do. They can come from a template with parameters…

In [ ]:
response = requests.get(provider + "/api/v1/catalog-agent/policy-templates", headers=json_header)
templates = show(response, "Policy templates", open_response=False)
table(
    templates,
    [
        ("Id", "id"),
        ("Version", "version"),
        ("Title", "title"),
        ("Parameters", lambda t: ", ".join(t.get("parameters") or {})),
        ("Actions", lambda t: ", ".join(p.get("action", "") for p in (t.get("content") or {}).get("permission", []))),
    ],
    title="Templates on the provider",
);

In [ ]:
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {"$bla": "did:web:university.upm.es"},
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants read access to did:web:university.upm.es.",
}
response = requests.post(provider + "/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, data=json.dumps(payload))
show(response, "Policy from the template");

…or be written directly:

In [ ]:
payload = {
    "odrlOffer": {
        "permission": [
            {
                "action": "use",
                "constraint": [
                    {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"}
                ],
            }
        ],
        "obligation": [
            {
                "action": "attribute",
                "constraint": [
                    {"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}
                ],
            }
        ],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial use: permitted with attribution; derivation prohibited.",
}
response = requests.post(provider + "/api/v1/catalog-agent/odrl-policies", headers=json_header, data=json.dumps(payload))
show(response, "Policy written directly");

## Connectors: how the provider reaches the factory

A connector template says how to talk to a kind of backend, with `{{__PARAMETERS__}}`; a connector instance fills them in for one distribution. The factory asks for an OAuth2 token, so the template carries the password grant: the provider fetches the token itself and the consumer never sees it.

The URLs are the ones the provider's agent uses, from its container.

In [ ]:
oauth2 = {
    "type": "OAUTH2",
    "grantType": {
        "type": "PASSWORD",
        "username": "{{__USER_NAME__}}",
        "password": {"type": "PLAIN", "content": "{{__USER_PASS__}}"},
    },
    "tokenUrl": "{{__TOKEN_URL__}}",
    "clientId": "{{__CLIENT_ID__}}",
    "clientSecret": {"type": "PLAIN", "content": "{{__CLIENT_SECRET__}}"},
    "scopes": [],
    "onTokenExpire": "REFRESH_OR_REFETCH",
}
oauth2_parameters = [
    {"paramType": "STRING", "name": "USER_NAME", "title": "User name", "required": True},
    {"paramType": "STRING", "name": "USER_PASS", "title": "User password", "required": True},
    {"paramType": "STRING", "name": "TOKEN_URL", "title": "Token URL", "required": True},
    {"paramType": "STRING", "name": "CLIENT_ID", "title": "Client ID", "required": True},
    {"paramType": "STRING", "name": "CLIENT_SECRET", "title": "Client secret", "required": True},
]
# The demo credentials, the same ones imported with the factory's Keycloak realm.
factory_credentials = {
    "USER_NAME": "testuser",
    "USER_PASS": "password",
    "TOKEN_URL": factory_token_url,
    "CLIENT_ID": "factory-client",
    "CLIENT_SECRET": "factory-secret",
}

### Pull

The agent calls the factory when the consumer pulls through the dataplane.

In [ ]:
payload = {
    "authentication": oauth2,
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}",
        },
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "URL", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True},
        *oauth2_parameters,
    ],
}
response = requests.post(provider + "/api/v1/connector/templates", headers=json_header, data=json.dumps(payload))
pull_template = show(response, "Pull connector template", open_response=False)

In [ ]:
payload = {
    "templateName": pull_template["name"],
    "templateVersion": pull_template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {"description": "GET /robots/ on the demo factory", "ownerId": "DEMO-FACTORY"},
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": factory_api + "/robots/",
        "ACCESS_METHODS": ["GET"],
        "ACCESS_HEADERS": {"Accept": "application/json"},
        **factory_credentials,
    },
}
response = requests.post(provider + "/api/v1/connector/instances", headers=json_header, data=json.dumps(payload))
show(response, "Pull connector instance, bound to the pull distribution", open_response=False);

### Push

When a push transfer starts, the agent subscribes to the factory with its own ingress URL (`{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}`, filled in at run time) and forwards every event to the consumer. When it ends, it unsubscribes with the id the factory returned (`{{__RUNTIME_JSON_{.subscription.data.ID}__}}`).

In [ ]:
payload = {
    "authentication": oauth2,
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {"Content-Type": "application/json"},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "{{__EVENT_TYPE__}}",
            }),
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({}),
        },
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription base URL", "required": True},
        {"paramType": "STRING", "name": "EVENT_TYPE", "title": "Event type", "required": True},
        *oauth2_parameters,
    ],
}
response = requests.post(provider + "/api/v1/connector/templates", headers=json_header, data=json.dumps(payload))
push_template = show(response, "Push connector template", open_response=False)

In [ ]:
payload = {
    "templateName": push_template["name"],
    "templateVersion": push_template["version"],
    "distributionId": distribution_push_id,
    "metadata": {"description": "Robot temperature events from the demo factory", "ownerId": "DEMO-FACTORY"},
    "parameters": {
        "SUB_URL": factory_events + "/subscriptions/subscribe",
        "UNSUB_URL": factory_events + "/subscriptions/unsubscribe",
        "EVENT_TYPE": "robot_temperature",
        **factory_credentials,
    },
}
response = requests.post(provider + "/api/v1/connector/instances", headers=json_header, data=json.dumps(payload))
show(response, "Push connector instance, bound to the push distribution", open_response=False);

## The consumer discovers the offer

The consumer asks for the provider's catalog over DSP, with its GNAP token. It gets every dataset the provider publishes, each with its distributions and policies.

In [ ]:
payload = {"associatedAgentPeer": provider_did, "filter": [], "noCache": True}
response = requests.post(consumer + "/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, data=json.dumps(payload))
remote_catalog = show(response, f"{consumer_name} requests {provider_name}'s catalog", open_response=False)["response"]

table(
    remote_catalog["dataset"],
    [
        ("Dataset", "@id"),
        ("Title", "title"),
        ("Formats", lambda d: ", ".join(x.get("format") or x.get("formats") or "?" for x in d.get("distribution") or [])),
        ("Policies", lambda d: len(d.get("hasPolicy") or [])),
        ("This run", lambda d: "◀" if d["@id"] == dataset_id else ""),
    ],
    title=f"{provider_name}'s catalog, as {consumer_name} sees it",
);

In [ ]:
dataset = next(d for d in remote_catalog["dataset"] if d["@id"] == dataset_id)
policy = dataset["hasPolicy"][0]
target_id = dataset["@id"]
policy_id = policy["@id"]
provider_dsp = remote_catalog["service"]["endpointURL"]


def rules(policy, kind):
    out = []
    for rule in policy.get(kind) or []:
        constraints = " and ".join(f"{c['leftOperand']} {c['operator']} {c['rightOperand']}" for c in rule.get("constraint") or [])
        out.append(rule.get("action", "") + (f" if {constraints}" if constraints else ""))
    return "; ".join(out) or "—"


table(
    dataset["hasPolicy"],
    [
        ("Policy", "@id"),
        ("Permission", lambda p: rules(p, "permission")),
        ("Obligation", lambda p: rules(p, "obligation")),
        ("Prohibition", lambda p: rules(p, "prohibition")),
        ("Chosen", lambda p: "◀" if p["@id"] == policy_id else ""),
    ],
    title=f"The offers on {dataset.get('title')}",
)
table(
    [{"what": "Dataset", "value": target_id}, {"what": "Policy", "value": policy_id}, {"what": "Provider DSP endpoint", "value": provider_dsp}],
    [("", "what"), ("", "value")],
);

## The consumer's callback address

Every DSP message names the address where the other side answers. The path depends on the DSP version; the host is the consumer as the provider reaches it from its container.

In [ ]:
response = requests.get(consumer + "/.well-known/dspace-version/2025-1", headers=json_header)
consumer_dsp = consumer_docker + show(response, "DSP 2025-1 on the consumer")["path"]
print(f"Consumer callback  {consumer_dsp}")